# Practice Lab 4.5 - RAG Failure Modes (WORKED solutions)

**Course:** Netsetos FDE Cohort 1 - Module 4 - Lesson 4.5 - Practice Lab

This notebook is the **worked-solution companion** to `practice-lab-4.5.html` (8 exercises).
Each exercise below repeats the task and success criteria, then shows a runnable worked
solution. Run it top to bottom.

- Exercises 1, 4, and 5 make Claude Sonnet calls. **No key needed to run:** the keyless guard
  in the next cell mocks the client so every cell executes end to end with canned text. Set
  `ANTHROPIC_API_KEY` to reproduce the real numbers quoted in the lesson.
- Exercise 3 uses Voyage `voyage-3.5` in production; the worked cell runs on deterministic
  pre-measured cosines so it executes keyless (no `VOYAGE_API_KEY`, no network).
- Exercises 2, 6, 7, and 8 are pure offline plumbing - they run and print their real output with no key.
- Production context: AcmeCorp lending-compliance RAG; GDPR / OCC / NIST AI RMF corpus; USD costs.
- Cost with a live key: well under the $0.40 lesson cap.

In [ ]:
!pip install --quiet anthropic openai

In [ ]:
# --- Keyless runnable guard (auto-inserted from the lesson HTML) --------------
# Learners WITH an ANTHROPIC_API_KEY / OPENAI_API_KEY hit the live model.
# WITHOUT a key, the common clients are mocked so this notebook still runs
# top-to-bottom with deterministic canned responses (no network, no cost).
import os
_HAS_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))
_HAS_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
if not (_HAS_ANTHROPIC or _HAS_OPENAI):
    print("No API key set -> KEYLESS MOCK mode (canned responses; set a key to go live).")
    try:
        import anthropic  # type: ignore
        class _MsgBlock:
            def __init__(self, text): self.text = text; self.type = "text"
        class _Msg:
            def __init__(self, text):
                self.content = [_MsgBlock(text)]
                self.stop_reason = "end_turn"
                self.usage = type("U", (), {"input_tokens": 0, "output_tokens": 0})()
        class _Messages:
            def create(self, **kw):
                return _Msg("[keyless-mock] set ANTHROPIC_API_KEY for a live response.")
        class _Anthropic:
            def __init__(self, *a, **k): self.messages = _Messages()
        anthropic.Anthropic = _Anthropic
        if hasattr(anthropic, "AsyncAnthropic"): anthropic.AsyncAnthropic = _Anthropic
    except Exception:
        pass
    try:
        import openai  # type: ignore
        class _OAMsg:
            def __init__(self, text): self.content = text
        class _OAChoice:
            def __init__(self, text): self.message = _OAMsg(text)
        class _OAResp:
            def __init__(self, text): self.choices = [_OAChoice(text)]
        class _Completions:
            def create(self, **kw):
                return _OAResp("[keyless-mock] set OPENAI_API_KEY for a live response.")
        class _Chat:
            def __init__(self): self.completions = _Completions()
        class _OpenAI:
            def __init__(self, *a, **k): self.chat = _Chat()
        openai.OpenAI = _OpenAI
        if hasattr(openai, "AsyncOpenAI"): openai.AsyncOpenAI = _OpenAI
    except Exception:
        pass
# ----------------------------------------------------------------------------

---
## Exercise 1 - Reproduce lost-in-the-middle on a 30K-token context (FM1)

**Task.** Build a 30K-token retrieved context by repeating a generic OCC circular paragraph. Insert a single needle - the AcmeCorp lending limit sentence - at position 5%, 50%, and 95% of the context. Ask Sonnet the same question 5 times per position and tally recall. Expect ~5/5 at 5% and 95%, ~2/5 or 3/5 at 50%.

**Success criteria**
- Context confirmed at ~30K tokens
- Needle inserted at exactly 5%, 50%, 95%
- 5 trials per position; recall tallied and printed
- Middle-position recall measurably lower than the ends (the U-shape)
- Total spend under $0.05 (15 Sonnet calls)

> Keyless mock returns one canned string for every call, so recall prints 0/5 at every position; set `ANTHROPIC_API_KEY` to see the real U-shape.

In [ ]:
import anthropic

client = anthropic.Anthropic()

# A boring filler paragraph from a generic OCC supervisory circular - repeat to fill 30K tokens.
FILLER = (
    "The OCC, in exercise of its supervisory authority, directs that all "
    "national banks shall maintain capital adequacy as per Basel III norms. "
    "This circular supersedes earlier guidance. "
)
NEEDLE = (
    "The AcmeCorp lending limit for unsecured loans to MSMEs in regional "
    "cities is USD 30,000 as of 13 Nov 2025."
)
QUESTION = "What is the AcmeCorp lending limit for unsecured MSME loans in regional markets?"

def build_context(needle_pos_pct: int, target_tokens: int = 30_000) -> str:
    # Approx 0.75 words per token for English; rebuild filler to hit budget.
    target_words = int(target_tokens * 0.75)
    filler_blob = (FILLER * 5000)[:target_words * 6]
    insert_at = int(len(filler_blob) * needle_pos_pct / 100)
    return filler_blob[:insert_at] + " " + NEEDLE + " " + filler_blob[insert_at:]

def ask(context: str) -> str:
    r = client.messages.create(
        model="claude-sonnet-5",
        max_tokens=120,
        system="Answer only from the provided context. If absent, say UNKNOWN.",
        messages=[{"role": "user",
                   "content": f"<ctx>{context}</ctx>\n\nQ: {QUESTION}"}],
    )
    return r.content[0].text

def test_position(pct: int, trials: int = 5) -> int:
    ctx = build_context(pct)
    hits = sum("30,000" in ask(ctx).lower() for _ in range(trials))
    print(f"position {pct:>3}% : {hits}/{trials} recall")
    return hits

for pos in (5, 50, 95):
    test_position(pos)
# Live-key shape: ~5/5 at 5%, ~2/5 at 50%, ~5/5 at 95% - the U-shape (Liu et al. 2023).
# Keyless mock returns one canned string, so recall prints 0/5 at every position.

---
## Exercise 2 - Dedup-by-canonical-doc-id pipeline (FM3 conflicting evidence)

**Task.** Define a `Doc` dataclass with `chunk_id`, `canonical_doc_id`, `effective_date`, `text`. Implement `dedup_by_canonical(docs)` that keeps the newest `effective_date` per `canonical_doc_id`. Demonstrate on a 5-doc set where 2 docs share `gdpr-art-7-consent`; assert 4 docs remain and the older one is gone.

**Success criteria**
- Dataclass carries canonical_doc_id + effective_date
- Dedup keeps newest per canonical id, drops the older version
- Assertion proves 5 docs in, 4 out, older GDPR chunk gone
- Stable order preserved (first-seen canonical id stays first)
- Total spend: $0 (no API calls)

In [ ]:
from dataclasses import dataclass
from datetime import date

@dataclass
class Doc:
    chunk_id: str
    canonical_doc_id: str
    effective_date: date
    text: str

def dedup_by_canonical(docs: list[Doc]) -> list[Doc]:
    """Keep newest effective_date per canonical_doc_id. Stable order."""
    seen: dict[str, Doc] = {}
    for d in docs:
        cur = seen.get(d.canonical_doc_id)
        if cur is None or d.effective_date > cur.effective_date:
            seen[d.canonical_doc_id] = d
    # Preserve first-seen order of canonical ids in the input.
    order = []
    written = set()
    for d in docs:
        cid = d.canonical_doc_id
        if cid not in written:
            order.append(seen[cid])
            written.add(cid)
    return order

retrieved = [
    Doc("c1", "gdpr-art-7-consent",  date(2023,  8, 11),
        "GDPR Art. 7 - consent definitions (original language)."),
    Doc("c2", "occ-master-kyc",      date(2024,  1,  4),
        "OCC bulletin on KYC v2 - PEP screening update."),
    Doc("c3", "gdpr-art-7-consent",  date(2025, 11, 13),
        "GDPR Art. 7 - clarified consent + withdrawal flow (EDPB 2025 note)."),
    Doc("c4", "nist-ai-rmf-confirm", date(2025,  8, 20),
        "NIST AI RMF - out-of-band confirm for AI-initiated transactions."),
    Doc("c5", "gdpr-art-17-erasure", date(2023,  8, 11),
        "GDPR Art. 17 - data subject right to erasure."),
]
deduped = dedup_by_canonical(retrieved)
assert len(deduped) == 4, f"expected 4, got {len(deduped)}"
for d in deduped:
    print(f"{d.canonical_doc_id:<22} {d.effective_date} {d.chunk_id}")
# Confirm the 2023 gdpr-art-7-consent chunk is gone
assert all(d.chunk_id != "c1" for d in deduped), "old GDPR consent not pruned"
print("\nFM3 dedup verified: older GDPR Art. 7 consent pruned.")

---
## Exercise 3 - FM4 off-topic retrieval filter - cosine-threshold gate

**Task.** Embed the erasure query and 8 candidate docs (on-topic GDPR + off-topic OCC capital-adequacy). Compute cosine per doc, apply a 0.55 threshold, print KEEP/DROP, and confirm the off-topic OCC chunks fall below the floor. Re-run at 0.70 to observe the recall trade-off.

**Success criteria**
- Embedder: voyage-3.5 (deterministic pre-measured cosines here so it runs keyless)
- Cosine per (query, doc) pair printed with KEEP / DROP verdict
- All 3 OCC chunks fall below the 0.55 floor (FM4 confirmed)
- Threshold sweep done: 0.55 vs 0.70, recall trade-off visible
- Total spend under $0.03 live (keyless here: $0)

In [ ]:
# Worked solution - FM4 cosine-floor off-topic filter.
# In production this embeds the query + 8 candidates with Voyage voyage-3.5 (the card
# starter) and computes cosine per pair. Here we use the pre-measured cosines so the
# cell runs keyless - no VOYAGE_API_KEY, no network - and reproduces the same verdicts.
QUERY = "What is the data subject's right to erasure under GDPR?"

# (candidate_id, cosine-to-query) measured with voyage-3.5 in the live lab.
CANDIDATE_COS = [
    ("gdpr-17-erasure",     0.812),
    ("gdpr-16-correction",  0.703),
    ("gdpr-17-timeline",    0.778),
    ("occ-capital-basel-3", 0.241),
    ("occ-liquidity-lcr",   0.198),
    ("gdpr-7-consent",      0.622),
    ("occ-kyc-pep",         0.273),
    ("gdpr-5-minimization", 0.598),
]

def filter_by_floor(threshold: float):
    print(f"\n--- threshold = {threshold} ---")
    kept, dropped = [], []
    for cid, score in CANDIDATE_COS:
        verdict = "KEEP" if score >= threshold else "DROP"
        print(f"  {verdict}  {score:.3f}  {cid}")
        (kept if score >= threshold else dropped).append(cid)
    print(f"  kept {len(kept)}, dropped {len(dropped)}")
    return kept, dropped

filter_by_floor(0.55)  # default: catches FM4 off-topic, keeps on-topic
filter_by_floor(0.70)  # tighter: prove the recall trade-off

# All 3 OCC capital-adequacy / PEP chunks fall below the 0.55 floor -> FM4 confirmed.
occ = {"occ-capital-basel-3", "occ-liquidity-lcr", "occ-kyc-pep"}
below = {cid for cid, s in CANDIDATE_COS if s < 0.55}
assert occ <= below, "all OCC chunks must fall below the 0.55 floor"
print("\nFM4 verified: every OCC chunk is below the 0.55 cosine floor.")

---
## Exercise 4 - Multi-hop decomposition (FM6) - compare X and Y

**Task.** Decompose "Compare GDPR and the NIST AI RMF on AI-system accountability obligations." into sub-questions with Sonnet, retrieve per sub-question against a stub corpus, aggregate, and synthesize the final compare answer. The multi-hop version grounds on both frameworks instead of one.

**Success criteria**
- Decompose returns at least 2 sub-questions (one per framework)
- Per-sub-question retrieval grounds on the right framework
- Final synthesis cites chunk ids from BOTH frameworks
- Baseline single-retrieval comparison shows the naive skew
- Total spend under $0.05

In [ ]:
import anthropic, json

client = anthropic.Anthropic()

CORPUS = [
    ("gdpr-accountability", "GDPR Art. 5(2): the data controller shall be accountable for compliance, including AI-driven decisions. Art. 35 requires a DPIA for high-risk processing."),
    ("gdpr-grievance",      "GDPR Art. 13: every data controller shall publish a contact point and respond to data-subject requests within about 30 days."),
    ("nist-ai-rmf-confirm",     "NIST AI RMF: financial transactions initiated by AI require out-of-band confirmation by the customer or a designated approver."),
    ("nist-ai-rmf-audit",       "NIST AI RMF (MANAGE function): AI-system decisions affecting credit shall be auditable; institutions shall log model version, input, output, and human reviewer (if any)."),
    ("nist-ai-rmf-explain",     "NIST AI RMF (MEASURE function): model decisions in credit and KYC shall include a human-readable explanation generated at decision time."),
    ("occ-kyc-pep",         "OCC KYC bulletin: PEP screening required at onboarding and annually."),
    ("gdpr-consent",        "GDPR Art. 6/7: processing of personal data requires consent or another lawful basis enumerated in Art. 6."),
]

def decompose(query: str) -> list[str]:
    """Ask the model to break a compare-style query into sub-questions."""
    r = client.messages.create(
        model="claude-sonnet-5",
        max_tokens=300,
        system="Decompose compare-style queries into 2-3 atomic sub-questions, one per entity. Return JSON list of strings only.",
        messages=[{"role": "user", "content": f"Query: {query}"}],
    )
    try:
        return json.loads(r.content[0].text)
    except json.JSONDecodeError:
        # keyless-mock (or any non-JSON reply): deterministic one-sub-question-per-framework split.
        return ["What does GDPR say about AI-system accountability?",
                "What does the NIST AI RMF say about AI-system accountability?"]

def retrieve_stub(sub_q: str, k: int = 3) -> list[tuple[str, str]]:
    """Toy retriever: substring match scored by token overlap."""
    q_toks = set(sub_q.lower().split())
    scored = [(len(q_toks & set(t.lower().split())), cid, t) for cid, t in CORPUS]
    scored.sort(reverse=True)
    return [(cid, t) for _, cid, t in scored[:k]]

def synthesize(query: str, contexts: list[tuple[str, list]]) -> str:
    blocks = []
    for sub_q, docs in contexts:
        chunks = "\n".join(f"  [{cid}] {t}" for cid, t in docs)
        blocks.append(f"Sub-question: {sub_q}\nRetrieved:\n{chunks}")
    payload = "\n\n".join(blocks)
    r = client.messages.create(
        model="claude-sonnet-5",
        max_tokens=500,
        system="Answer the original query using ONLY the retrieved chunks. Cite chunk ids inline.",
        messages=[{"role": "user", "content": f"{payload}\n\nOriginal: {query}"}],
    )
    return r.content[0].text

q = "Compare GDPR and the NIST AI RMF on AI-system accountability obligations."
subs = decompose(q)
print("Sub-questions:", subs)
ctxs = [(s, retrieve_stub(s)) for s in subs]
for s, docs in ctxs:
    print(f"\n  {s} -> {[cid for cid, _ in docs]}")
print("\nFinal answer:\n", synthesize(q, ctxs))
# Keyless: decompose() falls back to the 2 sub-questions above; synthesize() returns canned
# text. With a key you get the grounded compare answer citing BOTH frameworks.

---
## Exercise 5 - Citation-validation loop (FM7) - verify, re-prompt, fail closed

**Task.** Force strict-JSON answers `{"answer": ..., "citations": [...]}`. Parse the citations, intersect with the retrieved-set ids, and on a miss re-prompt with explicit feedback. Cap at 2 retries; if still invalid, return a fail-closed verdict. Test both the happy path and an induced-hallucination (sparse-retrieval) path.

**Success criteria**
- Strict JSON shape enforced; INSUFFICIENT_CONTEXT branch present
- Citation set intersected with retrieved ids; invalid citations identified
- Re-prompt feedback names the invalid citation AND the valid ids
- Retry cap of 2; fail-closed verdict on persistent hallucination
- Both happy + induced-hallucination paths tested
- Total spend under $0.05 (worst case 3 Sonnet calls per query)

In [ ]:
import anthropic, json
from dataclasses import dataclass, field

client = anthropic.Anthropic()

SYSTEM = """You answer GDPR / OCC compliance questions using only the retrieved chunks.

Strict output JSON:
{"answer": "<text>", "citations": ["<chunk-id>", ...]}

Rules:
- Every claim in the answer must be backed by at least one chunk-id from the retrieved set.
- chunk-ids must be drawn EXACTLY from the <chunks> block. Never invent ids.
- If the retrieved set does not support a confident answer, return
  {"answer": "INSUFFICIENT_CONTEXT", "citations": []}."""

@dataclass
class Verdict:
    ok: bool
    answer: str = ""
    citations: list[str] = field(default_factory=list)
    invalid: list[str] = field(default_factory=list)
    attempts: int = 0

def _chunks_block(retrieved: list[tuple[str, str]]) -> str:
    return "\n".join(f"<chunk id='{cid}'>{t}</chunk>" for cid, t in retrieved)

def ask_once(query: str, retrieved: list[tuple[str, str]], feedback: str = "") -> dict:
    user = f"<chunks>\n{_chunks_block(retrieved)}\n</chunks>\n\nQ: {query}"
    if feedback:
        user += f"\n\nFEEDBACK FROM PREVIOUS ATTEMPT: {feedback}"
    r = client.messages.create(
        model="claude-sonnet-5",
        max_tokens=400,
        system=SYSTEM,
        messages=[{"role": "user", "content": user}],
    )
    try:
        return json.loads(r.content[0].text)
    except json.JSONDecodeError:
        # Per the Ex 5 hint: malformed / keyless-mock JSON counts as insufficient context.
        return {"answer": "INSUFFICIENT_CONTEXT", "citations": []}

def answer_with_validation(query: str, retrieved: list[tuple[str, str]],
                           max_retries: int = 2) -> Verdict:
    valid_ids = {cid for cid, _ in retrieved}
    feedback = ""
    invalid: list[str] = []
    for attempt in range(max_retries + 1):
        resp = ask_once(query, retrieved, feedback)
        ans = resp.get("answer", "")
        cites = resp.get("citations", [])
        if ans == "INSUFFICIENT_CONTEXT":
            return Verdict(ok=True, answer=ans, citations=[], attempts=attempt + 1)
        invalid = [c for c in cites if c not in valid_ids]
        if not invalid:
            return Verdict(ok=True, answer=ans, citations=cites, attempts=attempt + 1)
        feedback = (f"citation(s) {invalid} were NOT in the retrieved set. "
                    f"Revise using only these ids: {sorted(valid_ids)}.")
    return Verdict(ok=False, answer="FAIL_CLOSED_HALLUCINATION", citations=[],
                   invalid=invalid, attempts=max_retries + 1)

retrieved = [
    ("gdpr-12", "GDPR section 12: data subject may request erasure when consent withdrawn."),
    ("gdpr-13", "GDPR section 13: grievance officer contact + 30-day response."),
]
print(answer_with_validation("What erasure rights does GDPR give?", retrieved))
# Sparse retrieval to provoke hallucination:
sparse = [("gdpr-13", "GDPR section 13: grievance officer + 30-day response.")]
print(answer_with_validation("What is the GDPR penalty for non-erasure?", sparse))
# Keyless: both return INSUFFICIENT_CONTEXT (mock text is not JSON). With a key the first
# validates on the first attempt and a hallucinated cite triggers the fail-closed path.

---
## Exercise 6 - Graded - categorize 50 failing answers, propose fixes, estimate uplift

**Task.** Read `data/failing_rag_50.csv`, add an `fm_label` column (FM1-FM8, FMUNK capped at 5), and ship `categorized_failures.csv` + a ~500-word `fix_memo.md` (FM frequency table, top-3 FMs with named fixes, per-fix hedged uplift, a diagnostic-flow runbook). +1 bonus: implement the highest-count fix end to end and report estimate vs actual.

**Success criteria**
- categorized_failures.csv has 50 rows, every row an fm_label, FMUNK <= 5
- fix_memo.md has the 4 named sections and is ~500 words
- Fix names: real libraries / thresholds / pattern names (no "improve X")
- Uplift estimates sum close to the dev-vs-prod gap, hedged honestly
- Bonus: one_fix_implemented.py reports estimate vs actual

> The 50-row CSV is your repo artifact. The worked cell demonstrates the categorize -> frequency-pivot -> top-3-prioritize pipeline on a self-contained authored 20-row sample so it runs keyless.

In [ ]:
# Worked solution - the categorize -> frequency-pivot -> prioritize pipeline the graded
# exercise asks for, on a self-contained authored 20-row sample (the 50-row CSV is your
# repo artifact; the same pivot runs on categorized_failures.csv).
import csv, io
from collections import Counter
from dataclasses import dataclass
from typing import Optional

@dataclass
class DiagRow:
    query_id: str
    query_text: str
    fm_primary: str
    fm_secondary: Optional[str]
    fix_lesson: str
    fix_status: str   # 'TODO' or 'DONE'
    notes: str

DIAG_ROWS: list[DiagRow] = [
    DiagRow('q-001', 'AcmeCorp retention rules que son para customer data?', 'FM8', 'FM4', 'L4.5+L4.1', 'TODO', 'code-mixed phrasing; English-only docs scored low'),
    DiagRow('q-002', 'Data-Protection Standard clause 4.2 exact text', 'FM3', None, 'L4.5', 'DONE', 'Two versions retrieved; canonical dedupe applied'),
    DiagRow('q-003', 'Compare retention rule with refund record-keeping', 'FM6', 'FM3', 'L4.5', 'TODO', 'Multi-hop; one side missing in retrieved set'),
    DiagRow('q-004', 'List grievance-redressal clauses', 'FM7', None, 'L4.5', 'TODO', 'Citations referenced docs not in retrieved set'),
    DiagRow('q-005', 'Refund window in the 2018 policy', 'FM5', None, 'L4.5', 'DONE', 'Stale 2018 doc surfaced; TTL filter added'),
    DiagRow('q-006', 'Default AcmeCorp retention window?', 'FM1', None, 'L4.5+L3.1', 'TODO', 'Needle at 50% of 30K context; recall dropped'),
    DiagRow('q-007', 'los datos del cliente, cuantos dias se pueden conservar?', 'FM8', None, 'L4.1', 'TODO', 'code-mixed; needs multilingual embedder'),
    DiagRow('q-008', 'Refund Policy Section 3.2 exact wording', 'FM2', None, 'L4.3', 'TODO', 'Dense missed exact section; needs hybrid + BM25'),
    DiagRow('q-009', 'Refund timeline for regulated-finance customers', 'FM6', 'FM2', 'L4.5+L4.3', 'TODO', 'Multi-hop AND irrelevant top-K context'),
    DiagRow('q-010', 'GDPR penalty schedule', 'FM7', 'FM4', 'L4.5', 'TODO', 'Hallucinated clause numbers; tightened citation rule'),
    DiagRow('q-011', 'Old 2019 retention policy still surfacing?', 'FM5', None, 'L4.5', 'DONE', 'Pre-2020 policy docs surfaced; TTL filter'),
    DiagRow('q-012', 'GDPR vs HIPAA retention difference', 'FM6', None, 'L4.5', 'TODO', 'Cross-jurisdiction multi-hop'),
    DiagRow('q-013', 'Default periodo de retencion de datos?', 'FM8', 'FM1', 'L4.1+L4.5', 'TODO', 'code-mixed AND long context'),
    DiagRow('q-014', 'Clause 4.2 amendment May 2025', 'FM3', None, 'L4.5', 'DONE', 'Conflicting versions resolved by canonical dedupe'),
    DiagRow('q-015', 'Government-ID verification docs needed', 'FM4', None, 'L4.5', 'TODO', 'Irrelevant top-K; cosine threshold added'),
    DiagRow('q-016', 'SOX Section 802 audit-trail exact wording', 'FM2', None, 'L4.3', 'TODO', 'Exact regulatory phrase; BM25 hybrid needed'),
    DiagRow('q-017', 'Customer-grievance escalation steps', 'FM7', None, 'L4.5', 'TODO', 'Citation hallucination on escalation steps'),
    DiagRow('q-018', 'Retention clause buried in a ~25K-token bundle', 'FM1', None, 'L4.5+L3.1', 'TODO', 'Needle lost in a long retrieved bundle'),
    DiagRow('q-019', 'Recent May 2025 retention amendment summary', 'FM3', 'FM5', 'L4.5', 'DONE', 'Newest version preferred; older purged by TTL'),
    DiagRow('q-020', 'Data Controller vs Data Processor', 'FM4', None, 'L4.5', 'TODO', 'Irrelevant billing docs surfaced; cosine threshold'),
]

# 1) emit categorized_failures.csv (here: to an in-memory buffer)
buf = io.StringIO()
w = csv.writer(buf)
w.writerow(['query_id', 'query_text', 'fm_primary', 'fm_secondary', 'fix_lesson', 'fix_status', 'notes'])
for r in DIAG_ROWS:
    w.writerow([r.query_id, r.query_text, r.fm_primary, r.fm_secondary or '', r.fix_lesson, r.fix_status, r.notes])
print("=== categorized CSV (first 6 lines) ===")
print("\n".join(buf.getvalue().splitlines()[:7]))

# 2) fix_memo.md section (a): FM frequency table (the pivot)
primary_counts = Counter(r.fm_primary for r in DIAG_ROWS)
print("\n=== pivot by fm_primary (Pareto) ===")
for fm, n in primary_counts.most_common():
    print(f"  {fm}: {n:>3} queries ({100*n/len(DIAG_ROWS):4.1f}%)  {'#' * n}")

# 3) fix_memo.md section (b/c): top-3 FMs -> named fix + hedged uplift
NAMED_FIX = {
    "FM8": "voyage-3.5 multilingual embedder + query-side language detect -> ~+8 pts (hedged)",
    "FM3": "canonical_doc_id + effective_date newest-wins dedup -> ~+14 pts (assumes clean dedup)",
    "FM6": "LLM query decomposition + per-sub retrieval + aggregate -> ~+6 pts (hedged)",
    "FM7": "force [doc_id] citations + validate against retrieved set + fail-closed -> ~+5 pts",
    "FM1": "anchor-at-end (L3.1) + rerank to top-5 (L4.3) -> ~+9 pts",
    "FM4": "cosine floor 0.55 on the embedding side -> ~+4 pts",
    "FM5": "TTL filter on effective_date (<=18mo unless query says historical) -> ~+3 pts",
    "FM2": "hybrid dense + BM25 for exact-phrase recall (L4.3) -> ~+4 pts",
}
top3 = [fm for fm, _ in primary_counts.most_common(3)]
print("\n=== top-3 FMs by count -> named fix (fix_memo.md sections b + c) ===")
for fm in top3:
    print(f"  {fm} ({primary_counts[fm]} rows): {NAMED_FIX[fm]}")

# 4) fix progress + recommended order
status_counts = Counter(r.fix_status for r in DIAG_ROWS)
print(f"\nfix progress: {dict(status_counts)}")
print(f"recommended fix order: {top3} "
      f"({sum(primary_counts[f] for f in top3)}/{len(DIAG_ROWS)} failing queries).")
print("Fix the top 3 first, re-eval, repeat. The spreadsheet IS the procurement-review artifact.")

---
## Exercise 7 - Bisect the failure: retrieval vs ranking vs generation

**Task.** Given 12 failing rows (offline), decide for each: is gold in top-50? in top-5? used in the answer? Route each to retriever-miss / reranker-demotion / generation-ignore and print the 3-bucket split. Localize WHERE the pipeline broke before you reach for an FM label.

**Success criteria**
- All 12 rows route to exactly one of the 3 buckets
- top5 asserted to be a subset of top50 (recall@5 <= recall@50)
- 3-bucket split prints a count per bucket, summing to 12
- You can name, per bucket, which owner / fix it routes to
- Total spend: $0 (offline)

In [ ]:
# ex7_bisect.py - localize each failure BEFORE you label it FM1-FM8.
# retriever-miss    : gold never made the top-50            -> fix chunking / embeddings / hybrid search
# reranker-demotion : gold in top-50, demoted out of top-5  -> fix the reranker / raise top-k
# generation-ignore : gold in top-5 but the answer skips it -> fix the prompt / anchor-at-end (FM1)
from collections import Counter

# 12 failing rows. `answer` cites chunk ids inline in [brackets]; top5 is a subset of top50.
FAILING = [
    # --- gold never retrieved (retriever-miss) ---
    {"id": "f01", "query": "erasure timeline under GDPR", "gold": "gdpr-17-timeline",
     "top50": ["gdpr-16-correction", "gdpr-7-consent", "occ-kyc-pep", "gdpr-5-minimization"],
     "top5":  ["gdpr-16-correction", "gdpr-7-consent"],
     "answer": "Correction is covered in [gdpr-16-correction]."},
    {"id": "f02", "query": "PEP screening frequency", "gold": "occ-kyc-pep",
     "top50": ["occ-liquidity-lcr", "occ-capital-basel-3", "gdpr-accountability"],
     "top5":  ["occ-liquidity-lcr", "occ-capital-basel-3"],
     "answer": "The LCR floor is in [occ-liquidity-lcr]."},
    {"id": "f03", "query": "out-of-band confirm for AI transactions", "gold": "nist-ai-rmf-confirm",
     "top50": ["nist-ai-rmf-audit", "nist-ai-rmf-explain", "gdpr-consent"],
     "top5":  ["nist-ai-rmf-audit", "nist-ai-rmf-explain"],
     "answer": "Audit logging is required per [nist-ai-rmf-audit]."},
    {"id": "f04", "query": "data minimization duty", "gold": "gdpr-5-minimization",
     "top50": ["gdpr-7-consent", "gdpr-16-correction", "occ-kyc-pep"],
     "top5":  ["gdpr-7-consent", "gdpr-16-correction"],
     "answer": "Consent basis is in [gdpr-7-consent]."},
    # --- gold retrieved into top-50 but demoted out of top-5 (reranker-demotion) ---
    {"id": "f05", "query": "right to erasure under GDPR", "gold": "gdpr-17-erasure",
     "top50": ["gdpr-16-correction", "gdpr-7-consent", "gdpr-17-erasure", "gdpr-5-minimization", "occ-kyc-pep"],
     "top5":  ["gdpr-16-correction", "gdpr-7-consent"],
     "answer": "Correction rights appear in [gdpr-16-correction]."},
    {"id": "f06", "query": "capital adequacy ratio floor", "gold": "occ-capital-basel-3",
     "top50": ["occ-liquidity-lcr", "occ-kyc-pep", "occ-capital-basel-3", "gdpr-accountability"],
     "top5":  ["occ-liquidity-lcr", "occ-kyc-pep"],
     "answer": "The LCR floor is in [occ-liquidity-lcr]."},
    {"id": "f07", "query": "AI decision explainability", "gold": "nist-ai-rmf-explain",
     "top50": ["nist-ai-rmf-audit", "nist-ai-rmf-confirm", "nist-ai-rmf-explain", "gdpr-accountability"],
     "top5":  ["nist-ai-rmf-audit", "nist-ai-rmf-confirm"],
     "answer": "Audit logs are required per [nist-ai-rmf-audit]."},
    {"id": "f08", "query": "controller accountability for AI decisions", "gold": "gdpr-accountability",
     "top50": ["gdpr-7-consent", "gdpr-grievance", "gdpr-accountability", "occ-kyc-pep"],
     "top5":  ["gdpr-7-consent", "gdpr-grievance"],
     "answer": "The grievance contact is in [gdpr-grievance]."},
    # --- gold sat in top-5 but the answer ignored it (generation-ignore) ---
    {"id": "f09", "query": "erasure when consent withdrawn", "gold": "gdpr-17-erasure",
     "top50": ["gdpr-17-erasure", "gdpr-7-consent", "gdpr-16-correction", "occ-kyc-pep"],
     "top5":  ["gdpr-17-erasure", "gdpr-7-consent"],
     "answer": "Consent is defined in [gdpr-7-consent]."},
    {"id": "f10", "query": "LCR minimum", "gold": "occ-liquidity-lcr",
     "top50": ["occ-liquidity-lcr", "occ-capital-basel-3", "occ-kyc-pep"],
     "top5":  ["occ-liquidity-lcr", "occ-capital-basel-3"],
     "answer": "The capital ratio floor is in [occ-capital-basel-3]."},
    {"id": "f11", "query": "audit log requirement", "gold": "nist-ai-rmf-audit",
     "top50": ["nist-ai-rmf-audit", "nist-ai-rmf-explain", "nist-ai-rmf-confirm"],
     "top5":  ["nist-ai-rmf-audit", "nist-ai-rmf-explain"],
     "answer": "Explanations are required per [nist-ai-rmf-explain]."},
    {"id": "f12", "query": "consent withdrawal flow", "gold": "gdpr-7-consent",
     "top50": ["gdpr-7-consent", "gdpr-17-erasure", "gdpr-16-correction"],
     "top5":  ["gdpr-7-consent", "gdpr-17-erasure"],
     "answer": "Erasure is covered in [gdpr-17-erasure]."},
]

def bisect(row: dict) -> str:
    gold = row["gold"]
    in50 = gold in row["top50"]
    in5  = gold in row["top5"]        # top5 subset of top50 => recall@5 <= recall@50
    used = gold in row["answer"]      # answers cite ids inline, so a substring hit == cited
    if not in50:
        return "retriever-miss"
    if not in5:
        return "reranker-demotion"
    if not used:
        return "generation-ignore"
    return "gold-used"               # gold present and cited: not a localization bucket

for row in FAILING:
    assert set(row["top5"]) <= set(row["top50"]), f"{row['id']}: top5 not within top50"
buckets = Counter()
for row in FAILING:
    b = bisect(row)
    buckets[b] += 1
    print(f"{row['id']}  gold={row['gold']:<18} -> {b}")
print("\n3-bucket split:")
for b in ("retriever-miss", "reranker-demotion", "generation-ignore"):
    print(f"  {b:<18} {buckets[b]}")
assert sum(buckets.values()) == len(FAILING)
assert buckets["gold-used"] == 0, "every failing row should localize to one of the 3 buckets"

---
## Exercise 8 - Flag failures on UNLABELED prod traffic (reference-free)

**Task.** Given 30 prod traces with NO expected answer, score each reference-free: weak-retrieval floor (~0.5), hedge-token count, missing-citation penalty, thumbs-down. Rank descending and print the top slice you would hand-triage - the sample Ex 6 assumes you already have. A USE_LLM flag routes to a claude-sonnet-5 judge in production; the default heuristic runs keyless.

**Success criteria**
- All 30 traces scored reference-free (no expected_answer read)
- Composite signal set present: weak-retrieval floor, hedge count, missing-citation, thumbs-down
- Output ranked descending; top slice (~2.0+) surfaced as the triage queue
- The code-mixed (FM8) low-cosine traces float to the top
- Total spend: $0 offline (LLM judge path optional)

In [ ]:
# ex8_flag_unlabeled.py - build the triage sample when nobody hands you labels.
# Ex 6 handed you 50 PRE-LABELED failures. Real prod traffic has no expected_answer.
# Production judge: a reference-free faithfulness + answer-relevance LLM scorer
#   (Ragas-style; Es et al. 2023, arxiv.org/abs/2309.15217). Set USE_LLM = True to
#   route each trace through claude-sonnet-5. The offline heuristic below runs keyless.
from collections import namedtuple

USE_LLM = False  # keep False to run offline / keyless

# 30 prod traces, no expected_answer. fields: id, query, max_cos (best retrieved-chunk
# cosine), answer, thumbs_down. A handful are code-mixed (FM8) prod queries.
Trace = namedtuple("Trace", "id query max_cos answer thumbs_down")
TRACES = [
    Trace("t01", "erasure rights under GDPR", 0.81, "Under GDPR Art. 17 a subject may request erasure. [gdpr-17-erasure]", False),
    Trace("t02", "cuota de prestamo si pago tarde", 0.32, "I'm not sure, this might depend on the loan terms.", True),
    Trace("t03", "lawful basis for processing", 0.77, "Consent or another lawful basis applies. [gdpr-consent]", False),
    Trace("t04", "penalty for late erasure", 0.41, "The penalty is unclear; possibly a fine.", False),
    Trace("t05", "PEP screening at onboarding", 0.74, "Enhanced due diligence is required for PEPs. [occ-kyc-pep]", False),
    Trace("t06", "reembolso tarde recargo", 0.29, "No estoy seguro. As far as I know there may be a fee.", True),
    Trace("t07", "LCR floor for banks", 0.79, "Banks maintain an LCR of at least 100 percent. [occ-liquidity-lcr]", False),
    Trace("t08", "capital buffer for small banks", 0.44, "It might be around 2.5 percent, I think.", False),
    Trace("t09", "audit fields to log", 0.72, "Log model version, input, output, reviewer. [nist-ai-rmf-audit]", False),
    Trace("t10", "does OCC require annual PEP", 0.66, "Possibly, but I cannot confirm from the documents.", True),
    Trace("t11", "right to correction", 0.76, "Inaccurate data may be corrected. [gdpr-16-correction]", False),
    Trace("t12", "erasure timeline days", 0.48, "The controller must erase, timeline unknown.", False),
    Trace("t13", "total capital ratio floor", 0.83, "The total capital ratio floor is set. [occ-capital-basel-3]", False),
    Trace("t14", "AI transaction approval", 0.39, "Unclear which rule applies here.", True),
    Trace("t15", "out-of-band confirm rule", 0.71, "AI-initiated transactions need out-of-band confirm. [nist-ai-rmf-confirm]", False),
    Trace("t16", "consentimiento GDPR", 0.35, "I think consent is needed, may depend on context.", False),
    Trace("t17", "data minimization duty", 0.69, "Controllers minimize data and limit purpose. [gdpr-5-minimization]", False),
    Trace("t18", "liquidity ratio minimum", 0.62, "The LCR minimum is possibly 100 percent.", True),
    Trace("t19", "grievance officer contact", 0.68, "A contact point is published. [gdpr-grievance]", False),
    Trace("t20", "grievance response window", 0.47, "Response within some days, unclear exactly.", False),
    Trace("t21", "explanation at decision time", 0.73, "A human-readable explanation is generated. [nist-ai-rmf-explain]", False),
    Trace("t22", "KYC onboarding documents", 0.58, "Enhanced due diligence for PEPs. [occ-kyc-pep]", True),
    Trace("t23", "DPIA requirement", 0.70, "A DPIA is required for high-risk processing. [gdpr-accountability]", False),
    Trace("t24", "audit fields required", 0.51, "Log model version, input, output. [nist-ai-rmf-audit]", False),
    Trace("t25", "erasure vs correction", 0.67, "Erasure in [gdpr-17-erasure], correction in [gdpr-16-correction].", False),
    Trace("t26", "prestamo mora interes", 0.30, "No documents found; I'm not sure.", True),
    Trace("t27", "capital adequacy Basel", 0.80, "Total capital ratio floor per [occ-capital-basel-3].", False),
    Trace("t28", "explainability at decision time", 0.45, "Explanations required, though details unclear.", False),
    Trace("t29", "lawful basis processing", 0.72, "Consent or another lawful basis. [gdpr-consent]", False),
    Trace("t30", "out of band confirm scope", 0.55, "It may depend on transaction type.", True),
]

HEDGES = ("not sure", "might", "possibly", "unclear", "cannot confirm",
          "as far as i know", "unknown", "i think", "may depend")

def judge_heuristic(t: Trace):
    """Reference-free failure score. Higher = more likely a failure worth triaging."""
    score, reasons = 0.0, []
    if t.max_cos < 0.5:                      # weak-grounding floor, about 0.5
        score += 2.0
        reasons.append(f"weak retrieval (max cosine {t.max_cos:.2f})")
    a = t.answer.lower()
    hedges = sum(h in a for h in HEDGES)
    if hedges:
        score += 0.6 * hedges
        reasons.append(f"{hedges} hedge token(s)")
    if "[" not in t.answer:                  # no chunk citation -> faithfulness risk
        score += 1.0
        reasons.append("no chunk citation")
    if t.thumbs_down:
        score += 1.5
        reasons.append("user thumbs-down")
    return score, reasons

def judge_llm(t: Trace):
    """Production path: a reference-free faithfulness + relevance judge via the API."""
    import anthropic
    client = anthropic.Anthropic()
    r = client.messages.create(
        model="claude-sonnet-5",
        max_tokens=10,
        system=("Score 0-3 how likely this RAG answer is WRONG or unfaithful, judging "
                "only faithfulness + relevance with NO gold answer. Reply one number."),
        messages=[{"role": "user", "content": f"Q: {t.query}\nA: {t.answer}"}],
    )
    return float(r.content[0].text.strip()), ["llm-judge"]

def score_all(traces):
    judge = judge_llm if USE_LLM else judge_heuristic
    scored = []
    for t in traces:
        s, reasons = judge(t)
        scored.append((round(s, 1), t.id, t.query, reasons))
    scored.sort(reverse=True)                # highest failure score first
    return scored

ranked = score_all(TRACES)
flagged = [r for r in ranked if r[0] >= 2.0]
print(f"{len(flagged)} of {len(TRACES)} traces flagged for hand-triage (score >= 2.0):\n")
for score, tid, query, reasons in ranked[:10]:
    mark = "FLAG" if score >= 2.0 else "ok  "
    print(f"  {mark} {score:4.1f}  {tid}  {query[:34]:<34} {'; '.join(reasons)}")

---
## Exercise 6 - grading rubric + submission checklist

**Grading rubric (10 points total, +1 bonus)**

| Axis | Weight | What full marks looks like |
|---|---|---|
| Categorization accuracy | 3 | At least 45 of 50 rows carry an FM1-FM8 label (FMUNK <= 5). Distribution roughly matches the AcmeCorp incident. Co-occurring FMs: dominant picked, secondary in notes. |
| Fix specificity | 3 | Each top-3 FM has a NAMED fix (voyage-3.5 + query-side language detect for FM8, not "improve retrieval"). Thresholds, library names, embedder versions spelled out. |
| Uplift estimate honesty | 2 | Each fix has a per-FM uplift number with a hedge. Total estimated uplift sums to the 62->88 gap; honest claims beat inflated ones. |
| Diagnostic flow doc | 2 | One paragraph the customer's eng team can run on the next regression: sampling protocol, FM checklist in priority order, per-FM diagnostic test, escalation path. |
| Bonus: one fix implemented end to end | +1 | Highest-count FM has a working fix; the 50-failure CSV is re-run; actual flips counted; estimate vs actual reported honestly. |

**Submission checklist**
- [ ] `categorized_failures.csv` - 50 rows, every row an `fm_label`, FMUNK <= 5
- [ ] `fix_memo.md` - the 4 named sections (frequency, top-3 fixes, uplift, diagnostic flow), ~500 words
- [ ] Fix names: real libraries / thresholds / pattern names (no "improve X")
- [ ] Uplift estimates sum close to the dev-vs-prod gap, hedged honestly
- [ ] Bonus: `one_fix_implemented.py` runs against the CSV; estimate vs actual reported
- [ ] Total spend under $0.20

**Feedback SLA** - submit in the cohort Discord `#practice-labs` channel; rubric-graded feedback is
returned within **24 hours** (Plus-tier mentor review). Self-paced learners use this worked notebook
and the rubric above to self-assess right away.